# Sanitized AMD GPU PyTorch Demo

这是一个独立、经过脱敏的 AMD GPU 工程示例。它只使用合成数据和标准小型神经网络，演示设备迁移、混合精度和性能测量；不包含任何未公开研究实现、数据或结果。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Please start Jupyter from the public_amd_demo directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from src.amd_runtime import (
    autocast_context, benchmark, choose_device, collect_runtime_info,
    memory_stats_mib, reset_peak_memory_if_needed, synchronize_if_needed,
)


## 1. 环境信息

ROCm 版 PyTorch 也使用 `torch.cuda` 命名空间作为兼容接口；这不代表正在使用 NVIDIA GPU。下面的单元会在无 GPU 时给出 CPU fallback 提示。

In [ ]:
runtime = collect_runtime_info()
for key, value in runtime.to_dict().items():
    print(f'{key}: {value}')

device = choose_device()
print(f'\nSelected device: {device}')
if device.type != 'cuda':
    print('Tip: run this notebook in an AMD ROCm PyTorch environment to collect AMD GPU metrics.')


## 2. 可复现性与合成数据

数据完全由随机种子在内存中生成，不下载或读取任何数据集。

In [ ]:
SEED = 2026
torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

sample_count, feature_count, class_count = 512, 32, 4
features_cpu = torch.randn(sample_count, feature_count)
labels_cpu = torch.randint(class_count, (sample_count,))
print('Synthetic feature shape:', tuple(features_cpu.shape))
print('Synthetic label shape:', tuple(labels_cpu.shape))


## 3. 通用模型和设备迁移

该模型是为本示例新写的标准两层 MLP，不关联任何研究模型。

In [ ]:
class TinyClassifier(torch.nn.Module):
    def __init__(self, input_features: int, output_classes: int) -> None:
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(input_features, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, output_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)

model = TinyClassifier(feature_count, class_count).to(device)
features = features_cpu.to(device)
labels = labels_cpu.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_function = torch.nn.CrossEntropyLoss()
print('Model and synthetic batch moved to:', device)


## 4. AMD GPU 训练步骤与混合精度

在 CUDA/ROCm 设备上启用 float16 autocast；CPU 上自动使用普通精度。异常信息会被保留给当前环境，而不会伪造成功结果。

In [ ]:
def training_step() -> tuple[float, torch.Tensor]:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with autocast_context(device, enabled=True):
        logits = model(features)
        loss = loss_function(logits, labels)
    loss.backward()
    optimizer.step()
    return float(loss.detach().cpu()), logits.detach()

try:
    first_loss, first_logits = training_step()
    print('First-step loss:', first_loss)
    print('Output shape:', tuple(first_logits.shape))
except RuntimeError as exc:
    print(f'Device execution failed: {exc}')
    print('Check your PyTorch/ROCm installation, then retry on CPU or a supported AMD GPU.')
    raise


## 5. 性能统计

先预热，再进行同步计时。下方不会写死任何性能数字；所有数值仅来自当前实际运行环境。

In [ ]:
reset_peak_memory_if_needed(device)

def measured_step() -> None:
    training_step()

try:
    stats = benchmark(measured_step, device, warmup_steps=5, measured_steps=20)
    stats['samples_per_second'] = sample_count / (stats['mean_latency_ms'] / 1000.0)
    print({key: round(value, 3) for key, value in stats.items()})
    gpu_memory = memory_stats_mib(device)
    print('Memory statistics (MiB):', gpu_memory if gpu_memory is not None else 'not available on CPU')
except RuntimeError as exc:
    print(f'Benchmark was not completed: {exc}')
    print('No performance result should be reported for this failed run.')


## 6. 结果展示与总结

本示例验证了通用的 AMD ROCm/PyTorch 适配流程：环境检测、`torch.cuda` 兼容接口、模型与数据迁移、可选混合精度、同步计时、显存统计，以及 CPU fallback。输出张量和损失仅来自合成分类任务，不代表任何研究实验结果。